# 01 — Explore Sentinel-1 STAC Catalog

Verify that we can search and download Sentinel-1 GRD scenes
from Microsoft Planetary Computer for our monitored sites.

In [ ]:
import sys
sys.path.insert(0, '../src')

from portvolume.config import load_config
from portvolume.acquisition.search import create_stac_client, search_scenes

config = load_config(config_dir='../config')
print(f'Loaded {len(config.sites)} sites ({len(config.ports)} ports, {len(config.airports)} airports)')

In [ ]:
# Search for Rotterdam scenes in January 2024
client = create_stac_client(config.stac_catalog_url)
rotterdam = config.get_site('rotterdam')

items = search_scenes(
    client=client,
    collection=config.collection,
    bbox=rotterdam.bbox,
    start_date='2024-01-01',
    end_date='2024-01-31',
)

print(f'Found {len(items)} scenes for Rotterdam in Jan 2024')
for item in items[:5]:
    print(f"  {item['id']}  {item['datetime']}  assets: {list(item['assets'].keys())}")

In [ ]:
# Download and display one scene
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import rioxarray

if items:
    item = items[0]
    vv_href = item['assets'].get('vv', {}).get('href')
    if vv_href:
        print(f'Loading VV band from: {vv_href[:80]}...')
        da = xr.open_dataarray(vv_href, engine='rasterio')
        
        # Clip to Rotterdam bbox
        w, s, e, n = rotterdam.bbox
        clipped = da.rio.clip_box(minx=w, miny=s, maxx=e, maxy=n)
        
        # Convert to dB for display
        linear = clipped.values.astype(np.float64)
        linear[linear <= 0] = 1e-10
        db = 10 * np.log10(linear)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(db.squeeze(), cmap='gray', vmin=-25, vmax=0)
        ax.set_title(f'Sentinel-1 VV (dB): Rotterdam — {item["datetime"]}')
        plt.colorbar(ax.images[0], label='σ₀ (dB)')
        plt.show()
        print(f'Shape: {clipped.shape}')

In [ ]:
# Check coverage across all sites for a month
from portvolume.acquisition.search import search_all_sites

all_results = search_all_sites(
    client=client,
    collection=config.collection,
    sites=config.sites[:5],  # First 5 sites only (to save time)
    start_date='2024-01-01',
    end_date='2024-01-31',
)

print('\nScene counts per site (Jan 2024):')
for site_id, items in all_results.items():
    print(f'  {site_id}: {len(items)} scenes')